# Resident Progress Classifier
**Business Question:** Which residents are progressing vs. stalling?
**Modeling Goal:** Predictive — flag at-risk residents so staff can intervene early.
**Explanatory:** Identify which factors drive stalling.


In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, recall_score, accuracy_score

import statsmodels.api as sm
import joblib
import warnings

warnings.filterwarnings('ignore')


## 1. Target Variable Definition
Deriving `risk_delta` to separate "Stalling" (Label 1) from "Progressing" (Label 0).


In [58]:
residents = pd.read_csv('lighthouse_csv_v7/residents.csv')

risk_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
residents['initial_risk_num'] = residents['initial_risk_level'].map(risk_map)
residents['current_risk_num']  = residents['current_risk_level'].map(risk_map)

residents['label'] = (residents['current_risk_num'] - residents['initial_risk_num']).apply(lambda x: 0 if x < 0 else 1)

print("Label Distribution:")
label_dist = residents['label'].value_counts(normalize=True)
print(label_dist)


Label Distribution:
label
1    0.533333
0    0.466667
Name: proportion, dtype: float64


## 2. Baseline Establishment
According to the textbook prediction guidelines, our **Baseline Accuracy** is the majority class proportion.


In [59]:
baseline_acc = label_dist.max()
majority_class = label_dist.idxmax()
print(f"Majority Class is Label {majority_class}")
print(f"Baseline Accuracy: {baseline_acc:.2%}")
print("Any model we train must surpass this baseline to be useful.")


Majority Class is Label 1
Baseline Accuracy: 53.33%
Any model we train must surpass this baseline to be useful.


## 3. Data Preparation & Feature Pruning
We aggressively filter highly sparse columns from `static_cols` to prevent overfitting.
We also drop `session_count` as it confounds `days_in_care`.


In [60]:
static_cols = [
    'resident_id', 'is_pwd', 'has_special_needs', 'family_is_4ps', 
    'family_solo_parent', 'family_indigenous', 'family_informal_settler'
]

residents['date_of_admission'] = pd.to_datetime(residents['date_of_admission'])
residents['days_in_care'] = (pd.to_datetime('today') - residents['date_of_admission']).dt.days

pr = pd.read_csv('lighthouse_csv_v7/process_recordings.csv')
state_score = {'Happy': 2, 'Hopeful': 1, 'Calm': 1, 'Distressed': -2, 'Angry': -1, 'Anxious': -1, 'Sad': -1, 'Withdrawn': -1}
pr['session_improvement'] = pr['emotional_state_end'].map(state_score) - pr['emotional_state_observed'].map(state_score)
counseling_features = pr.groupby('resident_id').agg(
    avg_session_duration   = ('session_duration_minutes', 'mean'),
    progress_rate          = ('progress_noted', 'mean'),
    avg_state_improvement  = ('session_improvement', 'mean')
).reset_index()

edu = pd.read_csv('lighthouse_csv_v7/education_records.csv')
education_features = edu.groupby('resident_id').agg(
    avg_attendance_rate  = ('attendance_rate', 'mean')
).reset_index()

health = pd.read_csv('lighthouse_csv_v7/health_wellbeing_records.csv')
health_features = health.groupby('resident_id').agg(
    avg_general_health  = ('general_health_score', 'mean')
).reset_index()

inc = pd.read_csv('lighthouse_csv_v7/incident_reports.csv')
inc['severity_num'] = inc['severity'].map({'Low': 1, 'Medium': 2, 'High': 3})
incident_features = inc.groupby('resident_id').agg(
    incident_count      = ('incident_id', 'count'),
    avg_severity        = ('severity_num', 'mean')
).reset_index()

ip = pd.read_csv('lighthouse_csv_v7/intervention_plans.csv')
plan_features = ip.groupby('resident_id').agg(
    pct_achieved = ('status', lambda x: (x == 'Achieved').mean())
).reset_index()

hv = pd.read_csv('lighthouse_csv_v7/home_visitations.csv')
hv['coop_score'] = hv['family_cooperation_level'].map({'Highly Cooperative': 2, 'Cooperative': 1, 'Neutral': 0, 'Uncooperative': -1})
visit_features = hv.groupby('resident_id').agg(
    avg_family_coop = ('coop_score', 'mean')
).reset_index()

from functools import reduce
dfs = [
    residents[static_cols + ['days_in_care', 'label']].set_index('resident_id'),
    counseling_features.set_index('resident_id'), education_features.set_index('resident_id'),
    health_features.set_index('resident_id'), incident_features.set_index('resident_id'),
    plan_features.set_index('resident_id'), visit_features.set_index('resident_id')
]

model_df = reduce(lambda a, b: a.join(b, how='left'), dfs).reset_index()

numeric_cols = model_df.select_dtypes(include=[np.number]).columns
model_df[numeric_cols] = model_df[numeric_cols].fillna(0)

print(f"Aggressively pruned dataset shape: {model_df.shape}")


Aggressively pruned dataset shape: (60, 18)


## 4. Pipeline & ColumnTransformer
Setting up rigid preprocessing ensuring NO data leakage during cross-validation grids.


In [61]:
X = model_df.drop(columns=['resident_id', 'label'])
y = model_df['label']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'bool']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## 5. Hyperparameter Tuning & Dynamic Model Selection
We dynamically track which model achieves the best Cross-Validated Recall score prioritizing the reduction of False Negatives.


In [62]:
pipelines = {
    'Logistic Regression': Pipeline([('pre', preprocessor), ('clf', LogisticRegression(random_state=42))]),
    'Decision Tree': Pipeline([('pre', preprocessor), ('clf', DecisionTreeClassifier(random_state=42))]),
    'Random Forest': Pipeline([('pre', preprocessor), ('clf', RandomForestClassifier(random_state=42))])
}

grids = {
    'Logistic Regression': {'clf__C': [0.01, 0.1, 1.0, 10.0]},
    'Decision Tree': {'clf__max_depth': [2, 3, 5], 'clf__min_samples_leaf': [2, 5]},
    'Random Forest': {'clf__max_depth': [2, 3, 5], 'clf__n_estimators': [50, 100]}
}

best_models = {}
best_cv_scores = {}

for name in pipelines.keys():
    gs = GridSearchCV(pipelines[name], grids[name], cv=cv, scoring='recall', n_jobs=-1)
    gs.fit(X, y)
    best_models[name] = gs.best_estimator_
    best_cv_scores[name] = gs.best_score_
    
    print(f"--- {name} ---")
    print(f"Best Params: {gs.best_params_}")
    print(f"Mean CV Recall: {gs.best_score_:.3f}")
    
    preds = best_models[name].predict(X)
    print(f"Overall Accuracy: {accuracy_score(y, preds):.3f} vs Baseline {baseline_acc:.3f}")
    print(f"F1 Score: {f1_score(y, preds):.3f}")

# Determine the dynamic winner based strictly on CV Recall
winning_name = max(best_cv_scores, key=best_cv_scores.get)
winning_pipeline = best_models[winning_name]

print(f"⭐ DYNAMIC WINNER: {winning_name} (CV Recall: {best_cv_scores[winning_name]:.3f})")


--- Logistic Regression ---
Best Params: {'clf__C': 0.01}
Mean CV Recall: 0.876
Overall Accuracy: 0.750 vs Baseline 0.533
F1 Score: 0.789
--- Decision Tree ---
Best Params: {'clf__max_depth': 5, 'clf__min_samples_leaf': 2}
Mean CV Recall: 0.538
Overall Accuracy: 0.917 vs Baseline 0.533
F1 Score: 0.915
--- Random Forest ---
Best Params: {'clf__max_depth': 2, 'clf__n_estimators': 100}
Mean CV Recall: 0.714
Overall Accuracy: 0.833 vs Baseline 0.533
F1 Score: 0.844
⭐ DYNAMIC WINNER: Logistic Regression (CV Recall: 0.876)


## 6. Exporting the Dynamically Chosen Model
The best performing model maximizing Recall during cross-validation is actively dumped to `resident-progress-classifier.pkl`.


In [63]:
# Export dynamically chosen pipeline
joblib.dump(winning_pipeline, 'resident-progress-classifier.pkl')
print(f"Dynamically exported {winning_name} to resident-progress-classifier.pkl!")

def get_prediction_confidence(resident_id: int, feature_df: pd.DataFrame) -> dict:
    model = joblib.load('resident-progress-classifier.pkl')
    row = feature_df[feature_df['resident_id'] == resident_id].drop(columns=['resident_id', 'label'], errors='ignore')
    
    prob = model.predict_proba(row)[0][1]
    
    # Threshold intentionally shifted to 0.4 prioritizing recall
    label = 'Stalling' if prob >= 0.4 else 'Progressing'
    return {'resident_id': resident_id, 'prediction': label, 'P(Stalling)': round(prob, 3)}

print("Demo Result:", get_prediction_confidence(model_df['resident_id'].iloc[0], model_df))


Dynamically exported Logistic Regression to resident-progress-classifier.pkl!
Demo Result: {'resident_id': np.int64(1), 'prediction': 'Stalling', 'P(Stalling)': np.float64(0.464)}
